# 06 - SymptoGraph-LLM, trained as an ablation ladder

Each rung adds **exactly one** novelty to the rung below it, so every reported
gain is attributable to a single component.

| Rung | Adds | Novelty |
|---|---|---|
| `L0_flat` | encoder with flat heads | - |
| `L1_bottleneck` | clinical symptom concept bottleneck | **N1** |
| `L2_graph` | symptom-disorder comorbidity graph | **N2** |
| `L3_weak` | confidence-weighted LLM weak labels | **N3** |
| `L4_contrastive` | symptom-profile contrastive objective | **N5** |
| `L5_full` | learned uncertainty and abstention | **N6** |

`L5_full` **is** SymptoGraph-LLM. `L0_flat` is deliberately the same model family
as the `mentalroberta` baseline from notebook 05, which anchors the ladder to a
number you already have.

**N4, the symptom-grounded rationale generator, is not in this notebook.** It is
a generative component needing its own training and human evaluation, so folding
it in here would conflate a classifier result with a generation result.

## How comparability is guaranteed

Metrics come from the identical `metrics.compute_metrics` function the baselines
use, on the identical splits from notebook 02, with the same seeds and the same
model-selection rule. Nothing in the reporting path differs.

## What gets written

Everything goes **straight to Drive** - no local staging, nothing to copy at the
end, so a dropped session loses nothing.

```
proposed/
  L0_flat/ L1_bottleneck/ L2_graph/ L3_weak/ L4_contrastive/ L5_full/
    seed_0/ seed_1/ seed_2/
      history.csv              every metric, train and val, appended on resume
      checkpoint_last.pt       model + optimiser + scheduler + RNG
      checkpoint_best.pt       best epoch by validation composite
      performance_metrics.json train/val/test + selective-prediction results
      config.json
      plots/                   9 figures at 300 dpi, font 20
    aggregate/                 mean +/- std over seeds, summary.csv, plots/
  comparison_vs_ablations/     the ladder, with per-rung deltas
  comparison_vs_baselines/     L5_full against notebook 05
```

## Budget

18 runs (6 rungs x 3 seeds). Roughly **4 hours on an A100**, longer on an L4.
Rungs L3 to L5 are slower because the weak labels enlarge the training set.
Everything is resumable and `RUNGS` lets you do a few rungs per session.

## 1. Mount Drive and configure

All paths point at Drive directly. Nothing is written to local disk.

In [ ]:
import os, sys, json, math, warnings
import pandas as pd
from google.colab import drive
warnings.filterwarnings('ignore')

drive.mount('/content/drive')

DRIVE_BASE = '/content/drive/MyDrive/hari/SymptoGraph-LLM'
PROCESSED = f'{DRIVE_BASE}/processed'          # notebook 02
GRAPH     = f'{DRIVE_BASE}/graph'              # notebook 03
WEAK      = f'{DRIVE_BASE}/weak_labels/weak_labels.parquet'   # notebook 04
BASELINES_DIR = f'{DRIVE_BASE}/baselines'      # notebook 05

# True -> tiny model and subset, about 3 minutes, separate folder
QUICK_TEST = True
SUFFIX = '_quicktest' if QUICK_TEST else ''
RESULTS = f'{DRIVE_BASE}/proposed{SUFFIX}'

# Which rungs to train in this session. None means all six.
RUNGS = None

os.makedirs(RESULTS, exist_ok=True)
print('mode    :', 'QUICK TEST' if QUICK_TEST else 'FULL RUN')
print('writing :', RESULTS)

missing = [p for p in (PROCESSED, GRAPH) if not os.path.isdir(p)]
assert not missing, f'missing prerequisites: {missing} - run notebooks 02 and 03'
print('processed:', sorted(os.listdir(PROCESSED)))
print('graph    :', sorted(os.listdir(GRAPH)))
print('weak     :', 'found' if os.path.exists(WEAK) else 'MISSING (run notebook 04)')
print('baselines:', 'found' if os.path.isdir(BASELINES_DIR) else 'not yet (notebook 05)')

## 2. Get the code

In [ ]:
REPO = 'https://github.com/sid-2k6/SymptoGraph-LLM.git'
CHECKOUT = '/content/SymptoGraph-LLM'

if os.path.isdir(f'{CHECKOUT}/.git'):
    !git -C $CHECKOUT fetch -q origin && git -C $CHECKOUT reset -q --hard origin/main
else:
    !git clone -q $REPO $CHECKOUT

need = ['src/symptograph/proposed.py', 'src/symptograph/metrics.py']
missing = [f for f in need if not os.path.isfile(f'{CHECKOUT}/{f}')]
assert not missing, f'stale checkout, missing {missing} - run: !rm -rf {CHECKOUT}'
sys.path.insert(0, f'{CHECKOUT}/src')
!git -C $CHECKOUT log --oneline -1

### GPU and batch size

Batch size is chosen from the available memory, since a Pro runtime may give you
anything from a T4 to an A100 and a larger batch is simply faster.

In [ ]:
import torch

if torch.cuda.is_available():
    gpu = torch.cuda.get_device_name(0)
    total_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    BATCH = 64 if total_gb > 60 else 32 if total_gb > 30 else 16
    print(f'{gpu}  {total_gb:.0f} GB  ->  batch size {BATCH}')
else:
    BATCH = 8
    print('No GPU. Runtime -> Change runtime type -> GPU, or expect this to crawl.')

## 3. The shared training protocol

One config object drives all 18 runs. Only the five component switches change
between rungs; every other hyperparameter is held fixed, and each rung's
`config.json` records exactly what it used.

Encoder and head use different learning rates (2e-5 and 1e-3) because the encoder
is pretrained and the heads are not. The `mentalroberta` baseline used the same
encoder rate, so `L0_flat` stays comparable to it.

In [ ]:
from symptograph.proposed import (ProposedConfig, run_ladder, LADDER,
                                  LADDER_ORDER, FULL_MODEL)
from symptograph.metrics import compare_runs, headline_table, set_publication_style
from symptograph.preprocess import Ontology

onto = Ontology.load(f'{CHECKOUT}/configs/symptom_ontology.json')
set_publication_style(font_size=20)

if QUICK_TEST:
    cfg = ProposedConfig(epochs=2, seeds=(0, 1, 2), batch_size=8,
                         eval_batch_size=16, max_length=64,
                         limit_train=400, limit_eval=200,
                         graph_dim=16, amp=False,
                         encoder_name='prajjwal1/bert-tiny',
                         weak_label_path=WEAK, weak_max_rows=200)
else:
    cfg = ProposedConfig(epochs=8, seeds=(0, 1, 2), batch_size=BATCH,
                         eval_batch_size=BATCH * 2, max_length=256,
                         encoder_name='mental/mental-roberta-base',
                         weak_label_path=WEAK,
                         weak_max_rows=20_000,
                         checkpoint_every=1)

print('epochs          ', cfg.epochs)
print('seeds           ', cfg.seeds)
print('batch           ', cfg.batch_size, '| max_length', cfg.max_length)
print('lr encoder/head ', cfg.lr, '/', cfg.head_lr)
print('task weights    ', cfg.task_weights)
print('weak rows cap   ', cfg.weak_max_rows)
print('selection       ', cfg.selection_metric, '(validation)')
print()
for name, flags in LADDER:
    on = [k.replace('use_', '') for k, v in flags.items() if v]
    print(f'  {name:16s} {", ".join(on) if on else "(none)"}')

## 4. Train the ladder

Each epoch prints **train and validation** side by side, so divergence is visible
immediately rather than at the end.

**If the session drops, re-run this cell.** Finished epochs are skipped and
`history.csv` is appended to, never rewritten. To split the work across sessions,
set `RUNGS` in cell 1, for example `RUNGS = ['L0_flat', 'L1_bottleneck']`.

In [ ]:
results = run_ladder(cfg, PROCESSED, GRAPH, RESULTS,
                     rungs=RUNGS, onto=onto, resume=True, verbose=True)
print('\ntrained:', list(results))

## 5. Epoch-wise history

`history.csv` carries about 310 columns: every per-symptom metric, every group
aggregate and every task metric, for **both** train and validation.

In [ ]:
TRAINED = [r for r in LADDER_ORDER
           if os.path.isfile(f'{RESULTS}/{r}/seed_0/history.csv')]

for rung in TRAINED:
    h = pd.read_csv(f'{RESULTS}/{rung}/seed_0/history.csv')
    print(f'\n=== {rung} (seed 0): {len(h)} epochs, {len(h.columns)} columns ===')
    cols = ['epoch', 'train_loss', 'val_loss',
            'train_group_all_macro_f1', 'val_group_all_macro_f1',
            'train_severity_depression_qwk', 'val_severity_depression_qwk',
            'train_composite', 'val_composite']
    display(h[[c for c in cols if c in h.columns]].round(4))

## 6. Publication figures for the full model

Every figure is 300 dpi with font size 20, ready to drop into the paper.

In [ ]:
from IPython.display import Image, display as show

plot_dir = f'{RESULTS}/{FULL_MODEL}/seed_0/plots'
if os.path.isdir(plot_dir):
    for p in sorted(os.listdir(plot_dir)):
        print(p)
        show(Image(filename=f'{plot_dir}/{p}', width=820))

## 7. Mean and standard deviation per rung

In [ ]:
for rung in TRAINED:
    meta = f'{RESULTS}/{rung}/aggregate/aggregate_metrics.json'
    if not os.path.isfile(meta):
        continue
    agg = json.load(open(meta))
    on = [k.replace('use_', '') for k, v in agg['components'].items() if v]
    print(f'\n=== {rung} === components: {", ".join(on) or "(none)"}'
          f' | best epoch per seed: {agg["per_seed_best_epoch"]}')
    display(headline_table(agg['aggregate'], label=rung)[['metric', 'formatted']]
            .set_index('metric'))

## 8. Comparison A - against the ablations

The ladder plot is the central ablation figure: each point adds one novelty, so a
rising line is the evidence that each component earns its place.

In [ ]:
ablation_runs = {r: f'{RESULTS}/{r}' for r in TRAINED}
combined_ab, tables_ab = compare_runs(
    ablation_runs, f'{RESULTS}/comparison_vs_ablations',
    title='SymptoGraph-LLM ablation ladder',
    ladder_order=LADDER_ORDER, font_size=20)

wide = combined_ab.pivot(index='metric', columns='model', values='formatted')
display(wide[[c for c in LADDER_ORDER if c in wide.columns]])

In [ ]:
# per-rung delta: what each novelty actually bought
rows = []
prev = None
for rung in TRAINED:
    t = tables_ab[rung].set_index('metric')
    cur = t['mean'].get('composite')
    rows.append({'rung': rung, 'composite': None if cur is None else round(cur, 4),
                 'delta_vs_previous': None if (cur is None or prev is None)
                                      else round(cur - prev, 4)})
    prev = cur if cur is not None else prev
deltas = pd.DataFrame(rows).set_index('rung')
display(deltas)
deltas.to_csv(f'{RESULTS}/comparison_vs_ablations/ladder_deltas.csv')

for p in sorted(os.listdir(f'{RESULTS}/comparison_vs_ablations')):
    if p.endswith('.png'):
        print(p)
        show(Image(filename=f'{RESULTS}/comparison_vs_ablations/{p}', width=880))

## 9. Comparison B - against the baselines

Kept deliberately separate from the ablation comparison: one answers *does the
architecture beat the prior art*, the other answers *does each component
contribute*. Mixing them into one table muddles both questions.

In [ ]:
from symptograph.baselines import BASELINES

baseline_runs = {b: f'{BASELINES_DIR}/{b}' for b in BASELINES
                 if os.path.isfile(f'{BASELINES_DIR}/{b}/aggregate/aggregate_metrics.json')}
if not baseline_runs:
    print(f'No baseline results in {BASELINES_DIR}. Run notebook 05 first '
          f'(with QUICK_TEST matching this notebook).')
else:
    runs = dict(baseline_runs)
    runs['SymptoGraph-LLM'] = f'{RESULTS}/{FULL_MODEL}'
    combined_bl, tables_bl = compare_runs(
        runs, f'{RESULTS}/comparison_vs_baselines',
        title='SymptoGraph-LLM vs baselines', font_size=20)
    display(combined_bl.pivot(index='metric', columns='model', values='formatted'))
    show(Image(filename=f'{RESULTS}/comparison_vs_baselines/comparison.png', width=900))

### Held-out test results

From each seed's **best-validation** checkpoint, so the test split never
influenced selection.

In [ ]:
test_rows = []
for label, path in {**{b: f'{BASELINES_DIR}/{b}' for b in baseline_runs},
                    'SymptoGraph-LLM': f'{RESULTS}/{FULL_MODEL}'}.items():
    meta = f'{path}/aggregate/aggregate_metrics.json'
    if os.path.isfile(meta):
        by = json.load(open(meta)).get('aggregate_by_split', {})
        if 'test' in by:
            test_rows.append(headline_table(by['test'], label=label))
if test_rows:
    tc = pd.concat(test_rows, ignore_index=True)
    display(tc.pivot(index='metric', columns='model', values='formatted'))
    tc.to_csv(f'{RESULTS}/comparison_vs_baselines/test_comparison.csv', index=False)

## 10. Selective prediction (N6)

Only `L5_full` has the uncertainty head. **AURC is lower-is-better**: it is the
area under the risk-coverage curve, so a falling curve means the model's
uncertainty is informative and deferring the least confident cases genuinely
reduces error - which is the operational claim behind N6.

In [ ]:
rows = []
for seed in cfg.seeds:
    p = f'{RESULTS}/{FULL_MODEL}/seed_{seed}/performance_metrics.json'
    if not os.path.isfile(p):
        continue
    sel = json.load(open(p)).get('selective', {}).get('test')
    if sel:
        rows.append({'seed': seed, 'aurc': round(sel['aurc'], 4),
                     'risk@100%': round(sel['risk_at_full_coverage'], 4),
                     'risk@80%': sel['risk_at_80_coverage'],
                     'risk@50%': sel['risk_at_50_coverage']})
if rows:
    sdf = pd.DataFrame(rows).set_index('seed')
    display(sdf)
    print('mean +/- std:')
    display(pd.DataFrame({'mean': sdf.mean().round(4),
                          'std': sdf.std(ddof=1).round(4)}))
    show(Image(filename=f'{RESULTS}/{FULL_MODEL}/seed_0/plots/risk_coverage.png',
               width=700))
else:
    print('No selective results yet - train L5_full.')

## 11. What to report

Everything is already on Drive. This cell just lists the artefacts.

In [ ]:
print('results root:', RESULTS)
!du -sh "$RESULTS"/* 2>/dev/null | sort -h
print()
lb = f'{RESULTS}/comparison_vs_ablations/leaderboard.md'
if os.path.isfile(lb):
    print(open(lb).read()[:1800])

## Notes

**Where each novelty lives in the code**, in `src/symptograph/proposed.py`:

- **N1** symptom-wise attention queries initialised from the sentence embeddings
  of the clinical definitions; severity, risk and stress are predicted *only*
  from the concept vector, so nothing bypasses the bottleneck.
- **N2** dense adjacency `A = A0 + dA`, where `A0` is notebook 03's graph and `dA`
  is learned under an L1 penalty. Inspect `model.delta` after training to report
  which edges the data added.
- **N3** weak labels enter as extra training rows whose per-symptom loss weight is
  the calibrated confidence from notebook 04.
- **N5** supervised contrastive loss, with positives defined by symptom-profile
  Jaccard overlap rather than a coarse class label.
- **N6** a per-symptom log-variance head with learned loss attenuation; that same
  variance is the abstention score in section 10.

**Reading the ladder honestly.** With 3 seeds, a gain smaller than the standard
deviation is not evidence. Suicide risk QWK in particular has only 340 training
and 74 test rows and swings widely. If a rung's delta is within noise, report it
as such and say so - a clean negative result on one component is far more
credible than six components that all conveniently help.

**A fairness point for N3.** Rungs L3 to L5 see weak labels that the baselines
never did. For the headline claim against prior art, also run notebook 05 with
`use_weak_labels=True` and compare against that, otherwise part of the gain is
extra data rather than architecture.

**Still to do for the full paper:** N4 (the symptom-grounded rationale generator
and its faithfulness evaluation), the concept-intervention experiment, and the
external evaluation on the held-out IMHI subsets.

### Troubleshooting

`Refusing to resume: ... came from a different architecture` means a checkpoint
in that folder was built with different settings, usually a quick test. The
message names each changed field; delete that run folder and start it again.
Quick-test output is kept in `proposed_quicktest/` for exactly this reason.

Checkpoints are full fp32 including optimiser state, about 1.5 GB each, written
to Drive every epoch. If that I/O dominates your epoch time, set
`checkpoint_every=2`.